# 37 Dressed-State Optimization

How dressed-state splitting shifts the dominant coherence constraint.

**Builds on:** `31_resonance_management.ipynb`

## Central question

> How does dressed-state splitting shift the dominant coherence constraint?

## Working statement

Dressed-state splitting shifts the operating regime from magnetic-noise sensitivity toward orbital and phononic loss channels, creating a useful coherence regime between those limits.

This notebook uses a toy model to make that tradeoff explicit.


## 1. Setup

We use a normalized dressed-state splitting:

\[
s \in [0, 1]
\]

where:

- lower `s` represents weak dressing and stronger magnetic-noise sensitivity
- intermediate `s` represents the useful coherence regime
- higher `s` represents stronger orbital / phononic loss channels

This is a qualitative model intended to organize the design space, not a fit to experimental data.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120


## 2. Regime model

The model has two main constraint terms.

### Magnetic-noise constraint

Magnetic-noise sensitivity decreases as dressed-state splitting increases.

### Orbital / phononic constraint

Orbital and phononic losses become more important at higher dressed-state splitting.

### Combined constraint

\[
C_{total}(s) = C_{magnetic}(s) + C_{phononic}(s)
\]

### Coherence proxy

\[
T_{proxy}(s) = \frac{1}{C_{total}(s)}
\]


In [ ]:
s = np.linspace(0.05, 1.0, 600)

def magnetic_constraint(s, amplitude=1.0, rate=4.2, floor=0.03):
    return amplitude * np.exp(-rate * s) + floor

def phononic_constraint(s, floor=0.10, threshold=0.50, curvature=3.0):
    excess = np.maximum(s - threshold, 0)
    return floor + curvature * excess**2

C_magnetic = magnetic_constraint(s)
C_phononic = phononic_constraint(s)
C_total = C_magnetic + C_phononic
T_proxy = 1 / C_total

opt_idx = np.argmax(T_proxy)
s_opt = s[opt_idx]
T_opt = T_proxy[opt_idx]
C_opt = C_total[opt_idx]

s_opt, T_opt, C_opt


## 3. Constraint curves

The useful operating regime appears where neither constraint dominates strongly.

At lower splitting, magnetic noise dominates.

At higher splitting, orbital / phononic constraints grow.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(s, C_magnetic, label="Magnetic-noise constraint")
ax.plot(s, C_phononic, label="Orbital / phononic constraint")
ax.plot(s, C_total, linewidth=2.5, label="Combined constraint")

ax.axvline(s_opt, linestyle="--", linewidth=1)
ax.scatter([s_opt], [C_opt], s=70, zorder=5)

ax.text(
    s_opt + 0.025,
    C_opt + 0.08,
    "useful operating regime",
    fontsize=10,
)

ax.set_title("Constraints vs Dressed-State Splitting")
ax.set_xlabel("Dressed-state splitting, normalized")
ax.set_ylabel("Relative constraint strength")
ax.legend()
ax.grid(True, alpha=0.3)

output = FIGURES / "37_constraints_vs_splitting.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 4. Coherence proxy

The coherence proxy increases as magnetic-noise sensitivity is reduced, then decreases as higher-splitting constraints dominate.

The maximum marks the model's useful operating regime.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(s, T_proxy, linewidth=2.5)
ax.axvline(s_opt, linestyle="--", linewidth=1)
ax.scatter([s_opt], [T_opt], s=80, zorder=5)

ax.text(
    s_opt + 0.025,
    T_opt * 0.92,
    f"optimum near s = {s_opt:.2f}",
    fontsize=10,
)

ax.set_title("Coherence Proxy vs Dressed-State Splitting")
ax.set_xlabel("Dressed-state splitting, normalized")
ax.set_ylabel("Coherence proxy, arbitrary units")
ax.grid(True, alpha=0.3)

output = FIGURES / "37_coherence_proxy.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 5. Dominant-constraint classification

Classify each splitting value by the dominant contribution.

This turns the toy model into a simple regime map.


In [ ]:
dominant = np.where(C_magnetic > C_phononic, "Magnetic-noise limited", "Orbital / phononic limited")

# Add a useful middle band near the coherence proxy optimum.
useful_band = np.abs(s - s_opt) < 0.08
dominant_with_middle = dominant.copy()
dominant_with_middle[useful_band] = "Useful coherence regime"

regime_df = pd.DataFrame(
    {
        "splitting": s,
        "magnetic_constraint": C_magnetic,
        "orbital_phononic_constraint": C_phononic,
        "combined_constraint": C_total,
        "coherence_proxy": T_proxy,
        "dominant_regime": dominant_with_middle,
    }
)

regime_df.head()


## 6. Regime map

The regime map is the main design-space output of this notebook.

It translates the continuous model into three operating regimes:

1. magnetic-noise-limited
2. useful coherence regime
3. orbital / phononic limited


In [ ]:
# Numeric encoding for display.
regime_order = {
    "Magnetic-noise limited": 0,
    "Useful coherence regime": 1,
    "Orbital / phononic limited": 2,
}

regime_values = np.array([regime_order[r] for r in dominant_with_middle])

fig, ax = plt.subplots(figsize=(10, 2.6))

ax.imshow(
    regime_values[np.newaxis, :],
    aspect="auto",
    extent=[s.min(), s.max(), 0, 1],
)

ax.axvline(s_opt, linestyle="--", linewidth=1)

ax.set_yticks([])
ax.set_xlabel("Dressed-state splitting, normalized")
ax.set_title("Dominant-Constraint Regime Map")

ax.text(0.18, 0.5, "magnetic-noise\nlimited", ha="center", va="center", fontsize=10)
ax.text(s_opt, 0.5, "useful\ncoherence", ha="center", va="center", fontsize=10, fontweight="bold")
ax.text(0.82, 0.5, "orbital / phononic\nlimited", ha="center", va="center", fontsize=10)

output = FIGURES / "37_regime_map.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 7. Parameter sensitivity

The location of the useful operating regime depends on how quickly magnetic sensitivity decreases and how quickly higher-splitting loss channels grow.

This cell sweeps two qualitative parameters:

- magnetic-noise suppression rate
- phononic-loss curvature

The output shows how the optimum shifts.


In [ ]:
rates = [2.5, 3.5, 4.5, 5.5]
curvatures = [1.5, 2.5, 3.5, 4.5]

rows = []

for rate in rates:
    for curvature in curvatures:
        Cm = magnetic_constraint(s, rate=rate)
        Cp = phononic_constraint(s, curvature=curvature)
        Ct = Cm + Cp
        Tp = 1 / Ct
        idx = np.argmax(Tp)
        rows.append(
            {
                "magnetic_suppression_rate": rate,
                "phononic_curvature": curvature,
                "optimal_splitting": s[idx],
                "max_coherence_proxy": Tp[idx],
            }
        )

sensitivity = pd.DataFrame(rows)
sensitivity


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for rate in rates:
    subset = sensitivity[sensitivity["magnetic_suppression_rate"] == rate]
    ax.plot(
        subset["phononic_curvature"],
        subset["optimal_splitting"],
        marker="o",
        label=f"magnetic rate {rate}",
    )

ax.set_title("Optimal Splitting Shifts with Constraint Strength")
ax.set_xlabel("Orbital / phononic curvature")
ax.set_ylabel("Optimal dressed-state splitting")
ax.grid(True, alpha=0.3)
ax.legend()

output = FIGURES / "37_optimum_sensitivity.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 8. Interpretation

This toy model supports the Notebook 31 roadmap.

Dressed-state splitting acts as an organizing variable because it shifts the active coherence constraint:

- lower splitting leaves magnetic fluctuations important
- intermediate splitting supports longer coherence
- higher splitting activates orbital / phononic constraints

The useful operating regime sits between the lower-splitting and higher-splitting limits.

Device-level improvements then determine whether that coherence regime can support fast pulses and cavity-compatible operation.


## 9. Next notebook

Notebook 41 can connect the optimized dressed-state regime to device constraints.

Candidate title:

> `41_fast_pulse_bandwidth.ipynb`

Candidate question:

> What pulse bandwidth is required to use the optimized dressed-state regime for noise suppression and cavity-compatible control?

Candidate outputs:

- pulse-duration vs bandwidth relation
- IDT bandwidth schematic
- efficiency / bandwidth tradeoff map
